# Multi-Model Text-to-Speech AI Inference — Guided API Demo / Demo API có hướng dẫn

### English

This notebook is a **guided API client** for an existing Text-to-Speech gateway: connect and authenticate, inspect model-specific settings, prepare a stable request, optionally submit an asynchronous job, then poll and play the WAV. The local CPU mock produces **silence**, not real synthesized voice. This is not a GPU benchmark or deployment notebook.

**Safety:** The default `SUBMIT_PAID_INFERENCE_JOB = False` sends **no POST**, downloads no model weights and launches no GPU. Enabling production submission may incur GPU charges.

### Tiếng Việt

Đây là **notebook hướng dẫn API client** của gateway Text-to-Speech đã có: kết nối và xác thực, xem tùy chọn của từng model, chuẩn bị request với định danh ổn định, tùy chọn gửi job bất đồng bộ, rồi theo dõi và phát WAV. CPU mock local chỉ tạo **âm thanh im lặng**, không phải giọng nói model thật. Đây không phải notebook benchmark GPU hoặc triển khai.

**An toàn:** Mặc định `SUBMIT_PAID_INFERENCE_JOB = False` sẽ **không POST**, không tải model weights và không chạy GPU. Nếu chủ động gửi job production, có thể phát sinh phí GPU.

**Workflow / Quy trình:** Connect / Kết nối → Model catalog / Danh mục model → Prepare / Chuẩn bị → Explicit submit / Chủ động gửi → Poll and listen / Theo dõi và nghe.


## Before you run / Trước khi chạy

### English

1. Open this file in a notebook editor (for example, VS Code with the Jupyter extension, or JupyterLab). From the repository root, install the client dependencies with `python -m pip install -e '.[notebook]'` and select that Python environment as the kernel. For a fresh JupyterLab setup, also run `python -m pip install jupyterlab` in that environment and launch `jupyter lab notebooks/tts_api_demo.ipynb`.
2. **Existing gateway:** obtain its HTTPS URL and a client Bearer token. Never save secrets to Git, the notebook source or screenshots. PostgreSQL and Redis are managed separately by the operator.
3. **Local CPU mock:** on the **same machine as the kernel**, install `python -m pip install -e '.[dev,notebook]'` and start `python scripts/local_demo.py` in a separate terminal. Use `http://127.0.0.1:8765` and the demo token printed by the server. WAV output is silent.
4. **Colab/Kaggle:** `127.0.0.1` is the remote notebook runtime, **not your laptop**. Connect to an existing HTTPS gateway reachable by the runtime.
5. **Costs:** Step 4 is the only cell that can submit a paid GPU job and defaults to off; verify model licenses and rights first.

### Tiếng Việt

1. Mở file bằng trình chạy notebook, ví dụ VS Code có tiện ích Jupyter hoặc JupyterLab. Từ thư mục gốc repo, chạy `python -m pip install -e '.[notebook]'` và chọn môi trường Python đó làm kernel. Nếu dùng JupyterLab lần đầu, cài thêm bằng `python -m pip install jupyterlab` trong cùng môi trường rồi chạy `jupyter lab notebooks/tts_api_demo.ipynb`.
2. **Gateway hiện có:** chuẩn bị URL HTTPS cùng Bearer token của client. Không lưu bí mật trong Git, mã nguồn notebook hoặc ảnh chụp. PostgreSQL và Redis do operator thiết lập riêng.
3. **CPU mock local:** trên **cùng máy chạy kernel**, cài `python -m pip install -e '.[dev,notebook]'`, rồi chạy `python scripts/local_demo.py` ở terminal khác. Dùng `http://127.0.0.1:8765` và token demo in trên terminal của server. WAV chỉ là im lặng.
4. **Colab/Kaggle:** `127.0.0.1` là runtime notebook từ xa, **không phải laptop**. Hãy dùng gateway HTTPS hiện có mà runtime có thể truy cập.
5. **Chi phí:** chỉ Bước 4 có thể gửi job GPU trả phí, mặc định đã tắt. Kiểm tra giấy phép và quyền sử dụng trước khi bật.


## Step 1 — Connect to an existing gateway / Kết nối gateway hiện có

### English

The next code cell imports the HTTP client, reads `TEXT_TO_SPEECH_API_BASE_URL` or prompts for a URL, and obtains `TEXT_TO_SPEECH_API_BEARER_TOKEN` or requests a hidden token. Remote endpoints require HTTPS; localhost HTTP is allowed for CPU mock only. **This cell does not submit audio.**

### Tiếng Việt

Cell code tiếp theo import HTTP client, đọc `TEXT_TO_SPEECH_API_BASE_URL` hoặc yêu cầu URL, rồi lấy `TEXT_TO_SPEECH_API_BEARER_TOKEN` hoặc hỏi token trong ô nhập ẩn. Endpoint từ xa bắt buộc HTTPS; chỉ cho phép HTTP trên localhost khi chạy CPU mock. **Cell này không gửi audio.**

### Expected result / Kết quả cần thấy

- `Gateway configured; token is not printed` / Đã cấu hình gateway và không hiển thị token.
- Invalid remote HTTP URL is rejected / URL HTTP từ xa không hợp lệ bị từ chối.


In [ ]:
import getpass
import hashlib
import json
import os
import time
import uuid
from urllib.parse import urlparse

import requests

TEXT_TO_SPEECH_GATEWAY_URL = (
    os.getenv("TEXT_TO_SPEECH_API_BASE_URL", "").rstrip("/")
    or input("Existing TTS gateway URL (https://...): ").rstrip("/")
)
parsed_gateway_url = urlparse(TEXT_TO_SPEECH_GATEWAY_URL)
is_local_http = (
    parsed_gateway_url.scheme == "http"
    and parsed_gateway_url.hostname in {"localhost", "127.0.0.1", "::1"}
)
if parsed_gateway_url.scheme != "https" and not is_local_http:
    raise ValueError("Use HTTPS remotely; only localhost may use HTTP for CPU mock")
CLIENT_BEARER_TOKEN = (
    os.getenv("TEXT_TO_SPEECH_API_BEARER_TOKEN", "")
    or getpass.getpass("Bearer token: ")
)
AUTHENTICATED_REQUEST_HEADERS = {"Authorization": f"Bearer {CLIENT_BEARER_TOKEN}"}
print("Gateway configured; token is not printed")


## Step 2 — Inspect the model catalog / Kiểm tra danh mục model

### English

This cell calls authenticated **`GET /v1/models`** and prints each model ID, `enabled` flag, checkpoint and `model_config` JSON Schema properties. MOSS-TTS-v1.5, Higgs-TTS-3 and VoxCPM2 have **different model controls**. Choose a model with `enabled: True` before submitting. This flag represents the operator’s selection. Higgs usually requires licensing approval. This read-only request **does not trigger GPU inference**.

### Tiếng Việt

Cell này gọi **`GET /v1/models` có xác thực**, in model ID, cờ `enabled`, checkpoint và JSON Schema của `model_config`. MOSS-TTS-v1.5, Higgs-TTS-3 và VoxCPM2 có **tham số riêng**. Chọn model có `enabled: True` trước khi gửi job. Cờ này thể hiện lựa chọn của người vận hành. Higgs thường cần xác nhận giấy phép. Request chỉ đọc này **không kích hoạt GPU**.

### Expected result / Kết quả cần thấy

- `enabled` and `checkpoint` for each model / Cờ hoạt động và checkpoint của mỗi model.
- `Controls:` lists model-specific options / Liệt kê tham số riêng của model.
- HTTP `401` means invalid credentials / Thiếu hoặc sai thông tin xác thực.


In [ ]:
model_catalog_response = requests.get(
    f"{TEXT_TO_SPEECH_GATEWAY_URL}/v1/models",
    headers=AUTHENTICATED_REQUEST_HEADERS,
    timeout=30,
)
model_catalog_response.raise_for_status()
for model in model_catalog_response.json()["data"]:
    print(model["id"], "enabled:", model["enabled"], "checkpoint:", model["checkpoint"])
    print(" Controls:", ", ".join(model["model_config_schema"]["properties"].keys()))


## Step 3 — Prepare an idempotent request / Chuẩn bị request ổn định

### English

Edit `SELECTED_MODEL_ID`, `TEXT_TO_SYNTHESIZE` and `MODEL_INFERENCE_OPTIONS`. The sample uses **VoxCPM2**; inspect Step 2's JSON Schema for other models. This cell creates a payload fingerprint and `Idempotency-Key`, but **never contacts the server**.

- To retry, keep the same generation, key, payload, gateway URL, and Bearer credential. Increase `REQUEST_GENERATION` only when deliberately starting a **new logical request**.
- Save the printed key and scope privately. After a kernel restart, set `TEXT_TO_SPEECH_REQUEST_ID` and `TEXT_TO_SPEECH_REQUEST_SCOPE` to those values, and restore the original gateway, credential, and payload before preparing. Clear both variables before starting a new request.
- If you know the job ID, prefer setting `TEXT_TO_SPEECH_EXISTING_JOB_ID` and polling with the original client on the original gateway. If credentials changed and the job ID is unknown, ask the operator to reconcile the original attempt.
- Deduplication lasts while the gateway retains the job. Confirm with the operator before replaying an old request. Restarting the CPU mock clears its jobs and keys.

### Tiếng Việt

Sửa `SELECTED_MODEL_ID`, `TEXT_TO_SYNTHESIZE` và `MODEL_INFERENCE_OPTIONS`. Ví dụ chọn **VoxCPM2**; xem JSON Schema ở Bước 2 nếu đổi model. Cell tạo fingerprint và `Idempotency-Key` nhưng **không gọi server**.

- Khi gửi lại, giữ nguyên generation, key, payload, URL gateway và token Bearer. Chỉ tăng `REQUEST_GENERATION` khi chủ động tạo **yêu cầu mới**.
- Lưu riêng key và giá trị scope được in ra. Sau khi khởi động lại kernel, đặt `TEXT_TO_SPEECH_REQUEST_ID` và `TEXT_TO_SPEECH_REQUEST_SCOPE` theo hai giá trị đó rồi khôi phục gateway, token và payload ban đầu trước khi chuẩn bị lại. Xóa cả hai biến trước khi tạo yêu cầu mới.
- Nếu đã có job ID, ưu tiên đặt `TEXT_TO_SPEECH_EXISTING_JOB_ID` và xem trạng thái bằng client ban đầu trên gateway ban đầu. Nếu token đã thay đổi và chưa có job ID, nhờ người vận hành đối soát lần gửi trước.
- Gateway chỉ chống gửi trùng trong thời gian còn lưu job. Hỏi người vận hành trước khi gửi lại yêu cầu cũ. Khởi động lại CPU mock sẽ xóa job và key.

### Expected result / Kết quả cần thấy

- `Prepared request key (save privately): ...` — new request / Request mới.
- `Existing logical request key: ...` — safe rerun / Chạy lại an toàn.
- `Read-only resume; no POST: ...` — resume / Tiếp tục job cũ.


In [ ]:
SELECTED_MODEL_ID = "voxcpm2"
TEXT_TO_SYNTHESIZE = "Xin chào! Đây là một ví dụ tổng hợp giọng nói bằng trí tuệ nhân tạo."
MODEL_INFERENCE_OPTIONS = {
    "gender": "female", "age": "young",
    "emotion": "cheerful", "pace": "slow",
    "cfg_value": 2.0, "inference_timesteps": 10,
}

# Keep this generation for retries; increment ONLY for an intentionally new paid job.
REQUEST_GENERATION = 1
RESUME_EXISTING_JOB_ID = os.getenv("TEXT_TO_SPEECH_EXISTING_JOB_ID", "").strip()
RESTORED_REQUEST_ID = os.getenv("TEXT_TO_SPEECH_REQUEST_ID", "").strip()
RESTORED_REQUEST_SCOPE = os.getenv("TEXT_TO_SPEECH_REQUEST_SCOPE", "").strip()
request_payload = {
    "model": SELECTED_MODEL_ID,
    "input": TEXT_TO_SYNTHESIZE,
    "model_config": MODEL_INFERENCE_OPTIONS,
}
request_fingerprint = hashlib.sha256(
    json.dumps(request_payload, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

def scoped_request_fingerprint(key, payload):
    scope = [
        TEXT_TO_SPEECH_GATEWAY_URL,
        AUTHENTICATED_REQUEST_HEADERS["Authorization"],
        key,
        payload,
    ]
    return hashlib.sha256(json.dumps(scope, sort_keys=True).encode()).hexdigest()


previous_request = globals().get("_PREPARED_TTS_REQUEST")
if RESUME_EXISTING_JOB_ID:
    SUBMITTED_JOB_ID = RESUME_EXISTING_JOB_ID
    print("Read-only resume; no POST:", SUBMITTED_JOB_ID)
elif previous_request and previous_request["generation"] == REQUEST_GENERATION:
    if previous_request["fingerprint"] != request_fingerprint:
        raise ValueError("Payload changed: increase REQUEST_GENERATION deliberately")
    STABLE_REQUEST_ID = previous_request["key"]
    if scoped_request_fingerprint(STABLE_REQUEST_ID, request_payload) != previous_request["scope"]:
        raise ValueError("The gateway or credentials changed; reconnect before retrying")
    if (RESTORED_REQUEST_ID or RESTORED_REQUEST_SCOPE) and (
        RESTORED_REQUEST_ID != STABLE_REQUEST_ID or RESTORED_REQUEST_SCOPE != previous_request["scope"]
    ):
        raise ValueError("Restored key and scope conflict with the prepared request")
    SUBMITTED_JOB_ID = previous_request.get("job_id")
    print("Existing logical request key:", STABLE_REQUEST_ID)
else:
    if previous_request and previous_request["acknowledgement"] == "uncertain":
        raise ValueError("Prior POST outcome uncertain; retry its original key first")
    if previous_request and (RESTORED_REQUEST_ID or RESTORED_REQUEST_SCOPE):
        raise ValueError("Clear TEXT_TO_SPEECH_REQUEST_ID and TEXT_TO_SPEECH_REQUEST_SCOPE before a new generation")
    STABLE_REQUEST_ID = RESTORED_REQUEST_ID or "nb-" + uuid.uuid4().hex
    request_scope = scoped_request_fingerprint(STABLE_REQUEST_ID, request_payload)
    if (RESTORED_REQUEST_ID or RESTORED_REQUEST_SCOPE) and (
        not RESTORED_REQUEST_ID or RESTORED_REQUEST_SCOPE != request_scope
    ):
        raise ValueError("Restore the original request scope, key, gateway, credential and payload together")
    _PREPARED_TTS_REQUEST = {
        "key": STABLE_REQUEST_ID,
        "payload": request_payload,
        "fingerprint": request_fingerprint,
        "generation": REQUEST_GENERATION,
        "scope": request_scope,
        "job_id": None,
        "acknowledgement": "uncertain" if RESTORED_REQUEST_ID else "not_submitted",
    }
    SUBMITTED_JOB_ID = None
    print("Prepared request key (save privately):", STABLE_REQUEST_ID)
    print("Request scope (save privately):", request_scope)

## Step 4 — Explicit submission and safe retry / Gửi job và retry an toàn

### English

**Only the following code cell can send `POST /v1/audio/jobs`.** Keep `SUBMIT_PAID_INFERENCE_JOB = False` for a safe `Run All`. Set it to `True` only after confirming the selected model, client authorization and potential GPU costs. A retry needs the **same key, payload, gateway and Bearer credential** within the retention window. Resume mode always skips POST.

- HTTP `202` with `job_id`: accepted; continue with Step 5.
- If the first attempt returns HTTP `403` (cloning policy), `422` (validation), or `503` with `Model disabled by deployment operator`, the gateway rejected it before admission. Correct the input or select an enabled model from Step 2, then increase `REQUEST_GENERATION`.
- Other `5xx` responses, `429`, timeouts, or a lost response can leave admission uncertain. Keep the original key and request details to retry or reconcile. A later rejection cannot resolve an earlier uncertain attempt.

### Tiếng Việt

**Chỉ code cell tiếp theo có thể gửi `POST /v1/audio/jobs`.** Giữ `SUBMIT_PAID_INFERENCE_JOB = False` để `Run All` an toàn. Chỉ đặt `True` sau khi xác nhận model, quyền client và khả năng phát sinh phí GPU. Khi gửi lại, dùng **cùng key, payload, gateway và token Bearer** trong thời gian gateway còn lưu job. Chế độ resume luôn bỏ qua POST.

- HTTP `202` kèm `job_id`: server đã nhận; tiếp tục Bước 5.
- Nếu lần gửi đầu trả HTTP `403` (chính sách clone giọng), `422` (dữ liệu không hợp lệ) hoặc `503` kèm `Model disabled by deployment operator`, gateway đã từ chối trước khi nhận job. Sửa dữ liệu hoặc chọn model đang bật ở Bước 2, rồi tăng `REQUEST_GENERATION`.
- Các lỗi `5xx` khác, `429`, timeout hoặc mất phản hồi có thể khiến trạng thái chưa rõ. Giữ key và thông tin yêu cầu ban đầu để gửi lại hoặc đối soát. Lỗi từ chối ở lần sau chưa xác nhận được kết quả lần gửi trước.

### Expected result / Kết quả cần thấy

- Default / Mặc định: `No POST sent. Enable submission only when you intend a paid job.`
- Opt-in / Chủ động gửi: `Job: ... reused: ...`.
- Acknowledged job / Job đã được xác nhận: `Already accepted; no POST. Poll job: ...`.
- Resume / Tiếp tục job cũ: `Resume mode: no POST. Poll job: ...`.


In [ ]:
# ONLY this cell sends POST. Opt in explicitly to billable inference.
SUBMIT_PAID_INFERENCE_JOB = False

if RESUME_EXISTING_JOB_ID:
    print("Resume mode: no POST. Poll job:", SUBMITTED_JOB_ID)
elif SUBMIT_PAID_INFERENCE_JOB:
    prepared = globals().get("_PREPARED_TTS_REQUEST")
    if not prepared or prepared["fingerprint"] != request_fingerprint:
        raise ValueError("Prepare a stable logical request first")
    if scoped_request_fingerprint(prepared["key"], prepared["payload"]) != prepared["scope"]:
        raise ValueError("The gateway or credentials changed, or the prepared payload was edited")
    if prepared["job_id"] is not None:
        SUBMITTED_JOB_ID = prepared["job_id"]
        print("Already accepted; no POST. Poll job:", SUBMITTED_JOB_ID)
    else:
        SUBMITTED_JOB_ID = None  # Never poll a stale job after a new failed POST.
        prior_acknowledgement = prepared["acknowledgement"]
        prepared["acknowledgement"] = "uncertain"
        print("Submitting/retrying with original request ID:", prepared["key"])
        try:
            submission_response = requests.post(
                f"{TEXT_TO_SPEECH_GATEWAY_URL}/v1/audio/jobs",
                headers={
                    **AUTHENTICATED_REQUEST_HEADERS,
                    "Idempotency-Key": prepared["key"],
                    "Content-Type": "application/json",
                },
                json=prepared["payload"],
                timeout=45,
            )
            submission_response.raise_for_status()
        except requests.HTTPError as error:
            response = error.response
            code = response.status_code if response is not None else None
            disabled_model = False
            if code == 503:
                try:
                    body = response.json()
                    disabled_model = (
                        isinstance(body, dict) and body.get("detail") == "Model disabled by deployment operator"
                    )
                except ValueError:
                    pass  # A proxy may return a non-JSON 503.
            if prior_acknowledgement != "uncertain" and (
                code in (403, 422) or disabled_model
            ):
                prepared["acknowledgement"] = "rejected"
                if disabled_model:
                    print("Model disabled: choose an enabled model from Step 2 and increment REQUEST_GENERATION")
                else:
                    print(f"HTTP {code}: rejected before admission; revise and increment REQUEST_GENERATION")
            else:
                print("Admission uncertain: retry with the same gateway, credential, key and payload")
            raise
        except requests.RequestException:
            print("ACK uncertain: retry with the same gateway, credential, key and payload")
            raise
        else:
            SUBMITTED_JOB_ID = submission_response.json()["job_id"]
            prepared["job_id"] = SUBMITTED_JOB_ID
            prepared["acknowledgement"] = "accepted"
            print("Job:", SUBMITTED_JOB_ID, "reused:", submission_response.json().get("reused"))
else:
    print("No POST sent. Enable submission only when you intend a paid job.")

## Step 5 — Poll status and listen to WAV / Theo dõi và nghe WAV

### English

The next cell uses **authenticated GET only** to poll an existing job, honors HTTP `429` and `Retry-After`, and downloads a WAV player once `completed`. Polling is time-bounded; this step **never submits a new job**. If polling ends while `queued` or `running`, save the job ID to resume.

- `completed`: display WAV and audio metadata.
- `failed`: inspect the job error before intentionally starting another request.
- `indeterminate`: uncertain GPU outcome requires **operator reconciliation**; never automatically resubmit.

### Tiếng Việt

Cell tiếp theo chỉ dùng **GET có xác thực** để kiểm tra job đã có, tuân thủ HTTP `429` và `Retry-After`, rồi tải WAV để phát khi `completed`. Polling có giới hạn thời gian; bước này **không tạo job mới**. Nếu dừng khi `queued` hoặc `running`, hãy lưu job ID để tiếp tục.

- `completed`: hiển thị WAV và metadata âm thanh.
- `failed`: xem lỗi trước khi chủ động tạo request khác.
- `indeterminate`: kết quả GPU chưa rõ, cần **operator đối soát**; không tự gửi lại.

### Expected result / Kết quả cần thấy

- `State: ...` displays job status / Hiển thị trạng thái job.
- On completion, WAV player and `Audio metadata:` / Khi hoàn tất, hiện trình phát WAV và metadata.
- On timeout, a `TEXT_TO_SPEECH_EXISTING_JOB_ID` restore hint / Khi hết thời gian, in gợi ý khôi phục job ID.
- The local CPU mock outputs silence / CPU mock local chỉ cho WAV im lặng.


In [ ]:
POLL_TIMEOUT_SECONDS = 300  # Increase for long queues or worker startup.
POLL_INTERVAL_SECONDS = 5
job_status_data = None

if SUBMITTED_JOB_ID:
    deadline = time.monotonic() + POLL_TIMEOUT_SECONDS
    while time.monotonic() < deadline:
        try:
            job_status_response = requests.get(
                f"{TEXT_TO_SPEECH_GATEWAY_URL}/v1/audio/jobs/{SUBMITTED_JOB_ID}",
                headers=AUTHENTICATED_REQUEST_HEADERS,
                timeout=30,
            )
            if job_status_response.status_code == 429:
                retry_header = job_status_response.headers.get("Retry-After", "5")
                try:
                    retry_seconds = max(1, min(300, int(retry_header)))
                except ValueError:
                    retry_seconds = 5
                print("Rate limited; retry status read after", retry_seconds, "seconds")
                time.sleep(min(retry_seconds, max(0, deadline - time.monotonic())))
                continue
            job_status_response.raise_for_status()
            job_status_data = job_status_response.json()
        except requests.RequestException as error:
            print("Status request interrupted; keep the job ID:", type(error).__name__)
            break
        print("State:", job_status_data["status"])
        if job_status_data["status"] in ("completed", "failed", "indeterminate"):
            break
        time.sleep(min(POLL_INTERVAL_SECONDS, max(0, deadline - time.monotonic())))

    if job_status_data and job_status_data["status"] == "completed":
        try:
            audio_download_response = requests.get(
                f"{TEXT_TO_SPEECH_GATEWAY_URL}{job_status_data['audio_url']}",
                headers=AUTHENTICATED_REQUEST_HEADERS,
                timeout=120,
            )
            audio_download_response.raise_for_status()
            from IPython.display import Audio, display
            display(Audio(data=audio_download_response.content, autoplay=False))
            print("Audio metadata:", job_status_data.get("audio"))
        except requests.RequestException as error:
            print("Audio download interrupted; keep the job ID:", type(error).__name__)
    elif job_status_data and job_status_data["status"] == "failed":
        print("Job failed; inspect its error before deliberately starting a new request")
    elif job_status_data and job_status_data["status"] == "indeterminate":
        print("GPU outcome uncertain; operator reconciliation required. Do not resubmit.")
    else:
        print("Job may still be queued/running; polling ended without a terminal result.")
        print('Restore with: os.environ["TEXT_TO_SPEECH_EXISTING_JOB_ID"] =', repr(SUBMITTED_JOB_ID))
else:
    print("No job ID. Explicitly submit or restore TEXT_TO_SPEECH_EXISTING_JOB_ID.")

## After the walkthrough / Sau khi thực hành

### English

- Voice attributes are **best effort**; an API demo is not a GPU-quality benchmark.
- Reference-voice cloning requires informed speaker permission **and operator approval for each client**. Self-declared `reference_consent=true` does not independently establish consent.
- Higgs-TTS-3 may have hosted/commercial licensing restrictions; see [model licenses](../MODEL_LICENSES.md).
- Clear notebook outputs before publication. Never paste Bearer tokens or private audio/text into Git.
- Read [API](../docs/API.md), [Deployment](../docs/DEPLOYMENT.md) and [Security](../docs/SECURITY.md) for the complete API/operator requirements.

### Tiếng Việt

- Thuộc tính giọng chỉ mang tính **gợi ý**; demo API không phải benchmark chất lượng GPU.
- Clone giọng tham chiếu phải có quyền của người nói **và operator cấp quyền theo client**. Cờ `reference_consent=true` do client tự khai không chứng minh consent.
- Higgs-TTS-3 có thể hạn chế quyền hosted/thương mại; xem [giấy phép model](../MODEL_LICENSES.vi.md).
- Xóa output trước khi công bố. Không đưa Bearer token hoặc audio/text riêng tư lên Git.
- Đọc [API](../docs/API.vi.md), [Triển khai](../docs/DEPLOYMENT.vi.md) và [Bảo mật](../docs/SECURITY.vi.md) để hiểu toàn bộ API và yêu cầu vận hành.
